# Notebook 1: Policy Agent - Tool Creation

## What This Notebook Does
This notebook creates the **Policy Agent** - a RAG (Retrieval Augmented Generation) agent that:
1. Uses **Databricks Vector Search** to find relevant policy documents
2. Answers questions about Foodly's official policies (refunds, cancellations, delivery, etc.)

## Architecture
```
User Question --> LLM --> Vector Search Tool --> Policy Documents --> LLM --> Answer
```

## Prerequisites
- A Databricks workspace with Vector Search enabled
- A Vector Search index `agents.main.foodly_policy_embedding_index` with policy documents
- Install required packages

In [ ]:
%pip install -U databricks-langchain langgraph mlflow[databricks]
dbutils.library.restartPython()

## Step 1: Import Dependencies and Configure LLM

We use `databricks-meta-llama-3-3-70b-instruct` - a **free** foundation model on Databricks.

In [ ]:
import json
from typing import Annotated, Any, Generator, Optional, Sequence, TypedDict, Union
from uuid import uuid4

import mlflow
from databricks_langchain import (
    ChatDatabricks,
    UCFunctionToolkit,
    VectorSearchRetrieverTool,
)
from langchain_core.language_models import LanguageModelLike
from langchain_core.messages import (
    AIMessage,
    AIMessageChunk,
    BaseMessage,
    convert_to_openai_messages,
)
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
)

## Step 2: Configure LLM and System Prompt

The Policy Agent has a focused system prompt - it ONLY answers policy questions.

In [ ]:
# Free Databricks Foundation Model
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

system_prompt = """You are the Foodly Policy Agent.
Answer ONLY questions about Foodly's official policies (refunds, cancellations, delivery, loyalty, privacy, fraud, escalation).
Always use the vector search tool to find the relevant section.
Summarize or quote faithfully; never guess, invent, or take actions.
If no policy is found, reply: "I couldn't find any official Foodly policy about that. Would you like me to escalate this?"
Keep responses clear, friendly, and precise"""

## Step 3: Create the Vector Search Retriever Tool

This tool connects to a **Databricks Vector Search Index** that contains Foodly's policy documents.

### How Vector Search Works:
1. Policy documents are chunked and embedded into vectors
2. When a user asks a question, the question is also embedded
3. Vector Search finds the most similar document chunks
4. The LLM uses these chunks to generate an accurate answer

**TODO:** Before running this, you need to:
1. Create a catalog `agents` and schema `main` in Unity Catalog
2. Upload policy documents to a Delta table
3. Create a Vector Search endpoint and sync the index

In [ ]:
tools = []

# TODO: Replace index_name with your actual Vector Search index
vs_tool = VectorSearchRetrieverTool(
    index_name="agents.main.foodly_policy_embedding_index",
    tool_name="foodly_policy_document_retrieval_tool",
    num_results=2,
    tool_description=(
        "Use this tool to search the Foodly knowledge base for policies, "
        "procedures, and service-related information. It retrieves the most "
        "relevant chunks from the company's official documentation, including "
        "refund rules, cancellation terms, delivery guidelines, loyalty program "
        "details, privacy policies, and escalation procedures"
    ),
)

VECTOR_SEARCH_TOOLS = [vs_tool]
tools.extend(VECTOR_SEARCH_TOOLS)

print(f"Tools configured: {[t.name for t in tools]}")

## Step 4: Create and Test the Agent

We use a helper function `create_tool_calling_agent` that builds a LangGraph workflow:
```
Entry --> Agent Node --> (has tool calls?) --> Yes --> Tools Node --> Agent Node
                                           --> No  --> END
```

In [ ]:
# Import from helpers module (must be in same directory)
from helpers import LangGraphResponsesAgent, create_tool_calling_agent

mlflow.langchain.autolog()
agent = create_tool_calling_agent(llm, tools, system_prompt)
AGENT = LangGraphResponsesAgent(agent)
mlflow.models.set_model(AGENT)

## Step 5: Test the Policy Agent

In [ ]:
for chunk in AGENT.predict_stream(
    {"input": [{"role": "user", "content": "What are foodly return policies?"}]}
):
    print(chunk.model_dump(exclude_none=True))